# PCA reusable template

**Short name:** `PCATel` pattern. Swap the CSV, the label column, and the audience nouns.

Pipeline: load → drop NA → split y → standardize → correlate → PCA → choose k → project → optional linear probe → simulate knobs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split

PATH = "data/telescope_data.csv"
LABEL = "class"
K_PLOT = 2
VAR_BUDGET = 0.95
TEST_SIZE = 0.33
SEED = 42

df = pd.read_csv(PATH, index_col=0).dropna()
y_raw = df[LABEL]
X_raw = df.drop(columns=[LABEL]).select_dtypes(include=np.number)
print(X_raw.shape, y_raw.value_counts().to_dict())

mu, sd = X_raw.mean(), X_raw.std(ddof=1).replace(0, 1.0)
X = (X_raw - mu) / sd

pca_all = PCA().fit(X)
cum = np.cumsum(pca_all.explained_variance_ratio_)
k_budget = int(np.searchsorted(cum, VAR_BUDGET) + 1)
print("k for", VAR_BUDGET, "=", k_budget, "cum=", cum[k_budget-1])

Z = PCA(n_components=K_PLOT).fit_transform(X)
y = y_raw.astype("category").cat.codes
Xtr, Xte, ytr, yte = train_test_split(Z, y, test_size=TEST_SIZE, random_state=SEED)
print("probe acc on", K_PLOT, "PCs:",
      LinearSVC(random_state=0, max_iter=8000).fit(Xtr, ytr).score(Xte, yte))
